# 3D Gaussian Splatting from DJI Avata 360 Drone Footage

Reconstruct a 3D scene from 360° drone video using Gaussian Splatting.

**Pipeline:** Equirectangular 360° video → Perspective extractions → COLMAP (SfM) → 3D Gaussian Splatting → Novel view rendering

**Input:** 108 perspective images extracted from Avata 360 (18 positions × 6 yaw angles)

**Hardware:** Colab A100 GPU required

In [ ]:
# Install dependencies
!pip install -q plyfile torch torchvision tqdm

# Install COLMAP
!apt-get -qq install colmap 2>&1 | tail -2

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

import os, glob, cv2
import numpy as np
import matplotlib.pyplot as plt
WORK_DIR = "/content/gaussian_splat"
os.makedirs(f"{WORK_DIR}/images", exist_ok=True)
print("Setup complete")


import zipfile, os, glob, cv2
from tqdm import tqdm

# Find images zip on Drive
zip_candidates = [
    "/content/drive/MyDrive/DroneCV/gaussian_splat/images.zip",
    "/content/drive/MyDrive/DroneCV/gaussian_splat/images_v2.zip",
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if not zip_path:
    from google.colab import files
    print("Not found on Drive. Upload images.zip:")
    uploaded = files.upload()
    zip_path = "/content/" + list(uploaded.keys())[0]
else:
    print(f"Found: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

print("Extracting...")
with zipfile.ZipFile(zip_path, "r") as z:
    members = z.namelist()
    print(f"  {len(members)} files")
    for m in tqdm(members, desc="Unzipping"):
        z.extract(m, WORK_DIR)

imgs = sorted(glob.glob(f"{WORK_DIR}/images/*.jpg"))
print(f"
✅ {len(imgs)} images ready")

# Show samples
fig, axes = plt.subplots(2, 6, figsize=(16, 5))
for i in range(6):
    img = cv2.cvtColor(cv2.imread(imgs[i]), cv2.COLOR_BGR2RGB)
    axes[0,i].imshow(img); axes[0,i].axis("off")
step = max(1, len(imgs)//6)
for i in range(6):
    img = cv2.cvtColor(cv2.imread(imgs[i*step]), cv2.COLOR_BGR2RGB)
    axes[1,i].imshow(img); axes[1,i].axis("off")
plt.suptitle(f"Input: {len(imgs)} perspective views from Avata 360")
plt.tight_layout(); plt.show()


In [ ]:
import zipfile
from google.colab import files

# Option A: Upload zip directly
zip_path = '/content/drive/MyDrive/DroneCV/gaussian_splat/images.zip'
if os.path.exists(zip_path):
    print(f'Found on Drive: {zip_path}')
elif os.path.exists('/content/images.zip'):
    zip_path = '/content/images.zip'
else:
    print('Upload images.zip:')
    uploaded = files.upload()
    zip_path = '/content/' + list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(WORK_DIR)

import glob
imgs = sorted(glob.glob(f'{WORK_DIR}/images/*.jpg'))
print(f'Extracted {len(imgs)} images')

# Show samples
import cv2
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(16, 5))
for i in range(6):
    img = cv2.cvtColor(cv2.imread(imgs[i]), cv2.COLOR_BGR2RGB)
    axes[0,i].imshow(img); axes[0,i].axis('off'); axes[0,i].set_title(f'Yaw {i*60}°')
for i in range(6):
    img = cv2.cvtColor(cv2.imread(imgs[54+i]), cv2.COLOR_BGR2RGB)
    axes[1,i].imshow(img); axes[1,i].axis('off')
plt.suptitle('Input: 18 positions × 6 views from Avata 360'); plt.tight_layout(); plt.show()

## Step 2: COLMAP — Structure from Motion

Estimate camera poses from the images using COLMAP's SfM pipeline.

In [ ]:
import subprocess

COLMAP_DIR = f'{WORK_DIR}/colmap'
DB_PATH = f'{COLMAP_DIR}/database.db'
SPARSE_DIR = f'{COLMAP_DIR}/sparse'
os.makedirs(SPARSE_DIR, exist_ok=True)

print('Running COLMAP feature extraction...')
subprocess.run(['colmap', 'feature_extractor',
    '--database_path', DB_PATH,
    '--image_path', f'{WORK_DIR}/images',
    '--ImageReader.single_camera', '1',
    '--ImageReader.camera_model', 'PINHOLE',
    '--SiftExtraction.max_image_size', '1024'],
    capture_output=True)

print('Running COLMAP feature matching...')
subprocess.run(['colmap', 'exhaustive_matcher',
    '--database_path', DB_PATH],
    capture_output=True)

print('Running COLMAP sparse reconstruction...')
result = subprocess.run(['colmap', 'mapper',
    '--database_path', DB_PATH,
    '--image_path', f'{WORK_DIR}/images',
    '--output_path', SPARSE_DIR],
    capture_output=True, text=True)

# Check result
sparse_models = glob.glob(f'{SPARSE_DIR}/*/images.bin')
if sparse_models:
    print(f'✅ COLMAP reconstruction successful: {len(sparse_models)} model(s)')
    # Convert to text format for inspection
    model_dir = os.path.dirname(sparse_models[0])
    subprocess.run(['colmap', 'model_converter',
        '--input_path', model_dir,
        '--output_path', model_dir,
        '--output_type', 'TXT'], capture_output=True)
    # Count registered images
    with open(f'{model_dir}/images.txt') as f:
        n_registered = sum(1 for line in f if line.strip() and not line.startswith('#')) // 2
    print(f'   Registered {n_registered}/{len(imgs)} images')
else:
    print('❌ COLMAP failed — try with more images or different matching')
    print(result.stderr[-500:] if result.stderr else 'No error output')

## Step 3: Train 3D Gaussian Splatting

Using gsplat (fast, PyTorch-native Gaussian splatting).

In [ ]:
# Simple Gaussian splatting training using gsplat
import numpy as np
import torch
from torch import nn, optim
from tqdm import tqdm
import struct

# Parse COLMAP output for camera poses
def parse_colmap_cameras(model_dir):
    """Parse COLMAP sparse reconstruction."""
    cameras = {}
    images_data = []
    
    # Parse cameras.txt
    with open(f'{model_dir}/cameras.txt') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.strip().split()
            if len(parts) >= 5:
                cam_id = int(parts[0])
                model = parts[1]
                w, h = int(parts[2]), int(parts[3])
                params = [float(p) for p in parts[4:]]
                cameras[cam_id] = {'model': model, 'w': w, 'h': h, 'params': params}
    
    # Parse images.txt
    with open(f'{model_dir}/images.txt') as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    
    for i in range(0, len(lines), 2):
        parts = lines[i].split()
        img_id = int(parts[0])
        qw, qx, qy, qz = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        tx, ty, tz = float(parts[5]), float(parts[6]), float(parts[7])
        cam_id = int(parts[8])
        name = parts[9]
        images_data.append({
            'name': name, 'qvec': [qw, qx, qy, qz],
            'tvec': [tx, ty, tz], 'cam_id': cam_id
        })
    
    return cameras, images_data

# Parse COLMAP points3D
def parse_colmap_points(model_dir):
    """Parse 3D points from COLMAP."""
    points = []
    colors = []
    with open(f'{model_dir}/points3D.txt') as f:
        for line in f:
            if line.startswith('#'): continue
            parts = line.strip().split()
            if len(parts) >= 7:
                points.append([float(parts[1]), float(parts[2]), float(parts[3])])
                colors.append([int(parts[4])/255, int(parts[5])/255, int(parts[6])/255])
    return np.array(points), np.array(colors)

model_dir = os.path.dirname(sparse_models[0])
cameras, images_data = parse_colmap_cameras(model_dir)
points_3d, point_colors = parse_colmap_points(model_dir)
print(f'COLMAP: {len(images_data)} images, {len(points_3d)} 3D points')
print(f'Point cloud bounds: {points_3d.min(axis=0)} to {points_3d.max(axis=0)}')

# Visualize point cloud
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
subsample = np.random.choice(len(points_3d), min(5000, len(points_3d)), replace=False)
ax.scatter(points_3d[subsample, 0], points_3d[subsample, 1], points_3d[subsample, 2],
           c=point_colors[subsample], s=1, alpha=0.5)
ax.set_title(f'COLMAP Sparse Point Cloud ({len(points_3d)} points)')
plt.tight_layout(); plt.show()

In [ ]:
# Initialize Gaussians from COLMAP points
N = len(points_3d)
print(f'Initializing {N} Gaussians from COLMAP point cloud...')

device = torch.device('cuda')
means = torch.tensor(points_3d, dtype=torch.float32, device=device, requires_grad=True)
colors = torch.tensor(point_colors, dtype=torch.float32, device=device, requires_grad=True)
scales = torch.full((N, 3), -5.0, device=device, requires_grad=True)  # log scale
opacities = torch.full((N,), 0.1, device=device, requires_grad=True)

print(f'Gaussians initialized on {device}')
print(f'  Means: {means.shape}')
print(f'  Colors: {colors.shape}')
print(f'  Scales: {scales.shape}')
print(f'\nTo fully train: use nerfstudio or the original gaussian-splatting repo')
print(f'This demo shows the reconstruction is feasible from Avata 360 footage')

## Results & Next Steps

**What we demonstrated:**
- DJI Avata 360 equirectangular → perspective extraction (18 positions × 6 views)
- COLMAP successfully reconstructs camera poses from 360° drone footage
- 3D point cloud of the scene recovered

**For full Gaussian Splatting training, use:**
```bash
# Clone official repo
git clone https://github.com/graphdeco-inria/gaussian-splatting
cd gaussian-splatting
python train.py -s /path/to/colmap/output --iterations 30000
```

**Applications:**
- GNSS-denied navigation: localize against the 3D Gaussian map
- Site inspection: photorealistic novel views of any location flown
- Change detection: compare splats from different dates
- Tactical awareness: full 3D model from a single 360° fly-over